# EPIC Clarity Procedure Occurrence Hydration


This notebook hydrates the OMOP PROCEDURE_OCCURRENCE table from EPIC Clarity order data.


## Source Table
- `_exponent._bronze_epic_clarity_*.dbo_ORDER_PROC`

In [ ]:
source = 'epic_clarity'

In [ ]:
-- Silver Layer: Transform EPIC procedure data%sqlCREATE OR REPLACE TEMP VIEW procedure_occurrence_silver ASSELECT    CONCAT_WS(CHR(31), 'epic_clarity', 'ORDER_PROC', 'ORDER_PROC_ID', op.ORDER_PROC_ID) AS procedure_source_value,    op.PROC_ID_PROC_NAME AS procedure_code_source_value,    op.ORDERING_DATE AS procedure_date,    COALESCE(pe.PAT_ENC_CSN_ID, NULL) AS visit_occurrence_source_value,    CURRENT_TIMESTAMP() AS updated_tspFROM _exponent._bronze_epic_clarity.order_proc_5 opLEFT JOIN _exponent._bronze_epic_clarity.pat_enc pe    ON op.PAT_ENC_CSN_ID = pe.PAT_ENC_CSN_IDWHERE op.ORDER_PROC_ID IS NOT NULL

In [ ]:
-- Merge into Silver Layer
%sql
MERGE INTO _exponent.omop_silver.procedure_occurrence AS target
USING procedure_occurrence_silver AS source
ON target.procedure_occurrence_source_value = source.procedure_occurrence_source_value

WHEN MATCHED AND NOT (
    target.procedure_source_value <=> source.procedure_source_value
)
THEN UPDATE SET
    target.updated_tsp = source.updated_tsp

WHEN NOT MATCHED THEN INSERT (
    procedure_occurrence_source_value,
    updated_tsp
)
VALUES (
    source.procedure_occurrence_source_value,
    source.updated_tsp
)

In [ ]:
-- Populate mapping table
%sql
INSERT INTO _exponent.omop_mapping.source_to_procedure_occurrence (
    source_system,
    procedure_occurrence_source_value,
    active_flag,
    created_tsp,
    last_mod_tsp,
    merge_id,
    merge_reason
)
SELECT
    'epic_clarity' AS source_system,
    s.procedure_occurrence_source_value,
    TRUE AS active_flag,
    CURRENT_TIMESTAMP() AS created_tsp,
    s.updated_tsp AS last_mod_tsp,
    NULL AS merge_id,
    NULL AS merge_reason
FROM (
    SELECT DISTINCT procedure_occurrence_source_value, updated_tsp
    FROM _exponent.omop_silver.procedure_occurrence
    WHERE procedure_occurrence_source_value IS NOT NULL
) s
LEFT ANTI JOIN _exponent.omop_mapping.source_to_procedure_occurrence x
    ON s.procedure_occurrence_source_value = x.procedure_occurrence_source_value
    AND x.source_system = 'epic_clarity'

In [ ]:
-- Gold Layer: Join with mapping to get procedure_occurrence_id
%sql
CREATE OR REPLACE TEMP VIEW procedure_occurrence_gold AS
SELECT
    m.procedure_occurrence_id,
    s.updated_tsp
FROM _exponent.omop_silver.procedure_occurrence s
INNER JOIN _exponent.omop_mapping.source_to_procedure_occurrence m
    ON s.procedure_occurrence_source_value = m.procedure_occurrence_source_value
    AND m.source_system = 'epic_clarity'
    AND m.active_flag = TRUE

In [ ]:
-- Merge into Gold Layer (OMOP)
%sql
MERGE INTO _exponent.omop.procedure_occurrence AS target
USING procedure_occurrence_gold AS source
ON target.procedure_occurrence_id = source.procedure_occurrence_id

WHEN NOT MATCHED THEN INSERT (
    procedure_occurrence_id
)
VALUES (
    source.procedure_occurrence_id
)